# Final checks for the second revision

| § | Question (reviewer concern) | What runs | Time |
|---|---|---|---|
| F1 | Is the harmonic-safe aperiodic fit (upper flank **45–55 Hz** instead of 32–45 Hz) a safe **default**? It fixes the +10 dB failure, but does it cost anything from −15 to +5 dB? (major 1) | The R1 sweep (real-data detector, three montages, five motor generators, six SNRs, 10 seeds, + 20 burst-free recordings) with **only** the aperiodic fit changed; the same seeds as R1, so every recording is compared pairwise with the primary fit | ~1.5 h |
| F2 | Do the real-data conclusions hold when the whole real-data analysis uses the harmonic-safe fit (gate, normalisation, threshold)? (major 1, 4) | All burst metrics, beta peak and slope, PD vs HC, OFF vs ON, meta-analysis, inclusion by group | ~15 min |
| F3 | Is the random-effects inference stable with k = 2–3 cohorts? (major 3) | Modified Knapp–Hartung (Röver et al., 2015) next to HKSJ for every meta-analysis in the paper | seconds |
| F4 | Does the two-generator conclusion depend on the 50 % / 20 mm criterion? (additional 1) | Two-generator cells of R3 (same seeds), "both imaged" at 30/50/70 % of the maximum and 15/20/25 mm | ~10 min |

Put this notebook in `real_data/` next to `revision_analyses.ipynb`, `pd_beta_three_datasets.ipynb` and
`pd_beta_three/`. It reuses their code and cached results and changes none of them. Outputs:
`pd_beta_three/revision/F*.csv`; every expensive step is cached.

**Paste back** the printed output of every cell.

In [1]:
"""Settings."""
F_DATASETS = None                  # None = all cohorts
F1_SNRS = (-15.0, -10.0, -5.0, 0.0, 5.0, 10.0)
F1_SEEDS = 10                      # ≤ 10 (R1's seeds)
F1_NOISE_ONLY = 20
F_METHODS = ["dSPM", "LCMV"]
HARMONIC_FLANK = (45.0, 55.0)      # V1: upper flank of the aperiodic fit (60 Hz notch, 100 Hz low-pass in preprocessing)
F4_LEVELS = (0.3, 0.5, 0.7)        # "second source imaged" if its local maximum ≥ level × global maximum
F4_RADII = (15.0, 20.0, 25.0)      # ... within this distance (mm) of the second source
REVISION_NOTEBOOK = "revision_analyses.ipynb"
RUN = {"F1": True, "F2": True, "F3": True, "F4": True}

In [2]:
"""Load the code of the revision notebook (settings, loader, R1 and R4 functions; nothing heavy runs)."""
import json
from pathlib import Path

_nb = json.loads(Path(REVISION_NOTEBOOK).read_text(encoding="utf-8"))
_want = ('"""Settings."""', '"""Load every definition', "def sim_record(", "def candidates(")
RESULTS_THR = {}
_RUN_FINAL = RUN
for _cell in _nb["cells"]:
    _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
    if _cell["cell_type"] == "code" and _src.lstrip().startswith(_want):
        exec(compile(_src, REVISION_NOTEBOOK, "exec"), globals())
RUN = _RUN_FINAL                   # the revision notebook's settings cell defines its own RUN
F_DATASETS = [d for d in (F_DATASETS or list(ANAT)) if d in ANAT]
_SPECTRAL_V0 = spectral
TRAP = getattr(np, "trapezoid", None) or np.trapz


def spectral_v1(sig, valid, fs):
    """spectral() of the main notebook with the upper flank of the aperiodic fit at 45–55 Hz (fit range 2–55 Hz)."""
    f, p = signal.welch(sig[valid], fs, nperseg=int(2 * fs))
    sel = (f >= 2) & (f <= HARMONIC_FLANK[1])
    f, p = f[sel], p[sel]
    flank = ((f >= 2) & (f <= 6.5)) | ((f >= HARMONIC_FLANK[0]) & (f <= HARMONIC_FLANK[1]))
    coef = np.polyfit(np.log10(f[flank]), np.log10(p[flank]), 1)
    fit = np.polyval(coef, np.log10(f))
    resid = 10 * (np.log10(p) - fit)
    b = (f >= 13) & (f <= 30)
    j = np.argmax(resid[b])
    return {"beta_db": float(resid[b][j]), "peak_hz": float(f[b][j]), "slope": float(coef[0]),
            "ap_var": float(TRAP(10 ** fit[b], f[b])), "curve": (f, 10 * np.log10(p), 10 * fit)}


def use_fit(name):
    """Every function of the pipeline (evaluate, choose_detector, normalised_beta) reads the global `spectral`."""
    global spectral
    spectral = spectral_v1 if name == "V1" else _SPECTRAL_V0


print(f"Cohorts: {[LABEL[d] for d in F_DATASETS]}")

Loaded pd_beta_three_datasets.ipynb: datasets ['sandiego', 'newmexico', 'iowa'], head model fsaverage, outputs → pd_beta_three\revision
  San Diego: 32 channels, simulated duration 183 s
  New Mexico: 60 channels, simulated duration 58 s
  Iowa: 58 channels, simulated duration 132 s
Cohorts: ['San Diego', 'New Mexico', 'Iowa']


## F1 — Simulation with the harmonic-safe fit as the default

Identical to R1 (real-data detector: flanking-band GED with the 5–11 + 32–45 Hz reference, gate, normalisation,
per-recording and cohort thresholds) except that the aperiodic fit used by the gate and the normalisation has its
upper flank at 45–55 Hz. The GED reference band is **not** changed, so this isolates the fit. Same seeds as R1:
every recording is compared with itself under the primary fit.

In [3]:
def f1_dataset(ds, fit="V1"):
    f = REV / f"F1_sim_{fit}_{ds}.csv"
    if f.exists():
        return pd.read_csv(f)
    use_fit(fit)
    a, cfg = ANAT[ds], {**CFG, "duration": DUR[ds]}
    jobs = [(reg, snr, 700_000 + 1000 * R1_REGIONS.index(reg) + 10 * list(R1_SNRS).index(snr) + k, True)
            for reg in MOTOR for snr in F1_SNRS for k in range(F1_SEEDS) if snr in R1_SNRS and reg in R1_REGIONS]
    jobs += [("L_M1_hand", 0.0, 790_000 + k, False) for k in range(F1_NOISE_ONLY)]
    first, pool, t0 = {}, [], time.time()
    try:
        for i, (reg, snr, seed, wb) in enumerate(jobs):
            X, ev, v = sim_record(a, reg, snr, seed, cfg, wb)
            det = choose_detector([X], [v], SFREQ, a["ch_names"], gate=True)
            sb, _ = normalised_beta(X, v, det["w"], SFREQ)
            first[seed] = {"w": det["w"], "passed": det["passed"], "reason": det["reason"], "slope": det["slope"],
                           "beta_db": det["beta_db"], "sm_frac": det["sm_frac"]}
            if det["passed"] and wb:
                e = np.abs(signal.hilbert(sb))[v]
                pool.append(np.random.default_rng(seed).choice(e, min(len(e), 20_000), replace=False))
            print(f"\r  {LABEL[ds]} {fit} pass 1: {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="")
        cohort_thr = float(np.percentile(np.concatenate(pool), THRESH_PERCENTILE))
        print(f"\n  cohort threshold {cohort_thr:.2f}")
        rows, t0 = [], time.time()
        for i, (reg, snr, seed, wb) in enumerate(jobs):
            X, ev, v = sim_record(a, reg, snr, seed, cfg, wb)
            d = first[seed]
            sb, _ = normalised_beta(X, v, d["w"], SFREQ)
            env = np.abs(signal.hilbert(sb))
            for scope, thr in (("per_recording", np.percentile(env[v], THRESH_PERCENTILE)), ("cohort", cohort_thr)):
                for r in score_run(a, X, v, ev, reg, sb, thr, cfg, methods=F_METHODS, oracle=False):
                    rows.append({"dataset": ds, "fit": fit, "region": reg, "snr_db": snr, "seed": seed, "bursts": wb,
                                 "scope": scope, "threshold": thr, "gate_passed": d["passed"], "gate_reason": d["reason"],
                                 "slope": d["slope"], "beta_db": d["beta_db"], "sm_frac": d["sm_frac"], **r})
            print(f"\r  {LABEL[ds]} {fit} pass 2: {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="")
        print()
    finally:
        use_fit("V0")
    df = pd.DataFrame(rows)
    df.to_csv(f, index=False)
    return df


if RUN["F1"]:
    # 0. the code path reproduces R1 under the primary fit (first three recordings of the first cohort)
    ds0 = F_DATASETS[0]
    r1f = REV / f"R1_sim_real_detector_{ds0}.csv"
    if r1f.exists():
        use_fit("V0")
        a, cfg = ANAT[ds0], {**CFG, "duration": DUR[ds0]}
        r1 = pd.read_csv(r1f)
        dev = []
        snr_c = 0.0 if 0.0 in R1_SNRS else float(R1_SNRS[0])
        for k in range(min(3, R1_SEEDS)):
            seed = 700_000 + 1000 * R1_REGIONS.index("L_M1_hand") + 10 * list(R1_SNRS).index(snr_c) + k
            X, ev, v = sim_record(a, "L_M1_hand", snr_c, seed, cfg, True)
            det = choose_detector([X], [v], SFREQ, a["ch_names"], gate=True)
            sb, _ = normalised_beta(X, v, det["w"], SFREQ)
            thr = np.percentile(np.abs(signal.hilbert(sb))[v], THRESH_PERCENTILE)
            for r in score_run(a, X, v, ev, "L_M1_hand", sb, thr, cfg, methods=["dSPM"], oracle=False):
                ref = r1[(r1.seed == seed) & (r1.scope == "per_recording") & (r1.method == "dSPM")]
                if len(ref):
                    dev.append(abs(r["peak_err_mm"] - ref.peak_err_mm.iloc[0]))
        print(f"Code check: primary fit reproduces R1, max |Δ error| = {max(dev) if dev else float('nan'):.2f} mm (0 = identical)\n")

    F1 = pd.concat([f1_dataset(ds) for ds in F_DATASETS], ignore_index=True)
    R1o = pd.concat([pd.read_csv(REV / f"R1_sim_real_detector_{ds}.csv") for ds in F_DATASETS], ignore_index=True)
    R1o = R1o[R1o.method.isin(F_METHODS) & R1o.scope.isin(["per_recording", "cohort"])]
    keys = ["dataset", "seed", "scope", "method"]
    cols = ["peak_err_mm", "region_correct", "f1", "gate_passed", "n_detected", "time_in_burst", "slope"]
    P = F1.merge(R1o[keys + cols], on=keys, suffixes=("_V1", "_V0"))
    print(f"Paired recordings × scopes × methods: {len(P)} (of {len(F1)})")

    B = P[P.bursts.astype(bool) & (P.scope == "cohort")]
    out = []
    for (ds, snr, m), g in B.groupby(["dataset", "snr_db", "method"]):
        gp0, gp1 = g.gate_passed_V0.astype(bool), g.gate_passed_V1.astype(bool)
        e0, e1 = g.loc[gp0, "peak_err_mm_V0"], g.loc[gp1, "peak_err_mm_V1"]
        both = gp0 & gp1
        d = (g.loc[both, "peak_err_mm_V1"] - g.loc[both, "peak_err_mm_V0"]).dropna()
        p = stats.wilcoxon(d).pvalue if len(d) > 5 and np.any(d != 0) else np.nan
        out.append({"dataset": LABEL[ds], "snr_db": snr, "method": m, "n": len(g),
                    "gate_V0": gp0.mean(), "gate_V1": gp1.mean(),
                    "err_V0": e0.median(), "err_V1": e1.median(),
                    "acc_V0": g.loc[gp0, "region_correct_V0"].mean(), "acc_V1": g.loc[gp1, "region_correct_V1"].mean(),
                    "F1_V0": g.loc[gp0, "f1_V0"].mean(), "F1_V1": g.loc[gp1, "f1_V1"].mean(),
                    "paired_median_diff_mm": d.median() if len(d) else np.nan, "wilcoxon_p": p})
    S = save(pd.DataFrame(out), "F1_summary")
    print("Primary fit (V0, 32–45 Hz flank) vs harmonic-safe fit (V1, 45–55 Hz) — cohort threshold, five motor generators, "
          "gated recordings. Paired difference V1 − V0 over recordings gated under both (negative = V1 better):")
    show(S, nd=2)

    print("\nPooled over cohorts (cohort threshold):")
    pooled = []
    for (snr, m), g in B.groupby(["snr_db", "method"]):
        gp0, gp1 = g.gate_passed_V0.astype(bool), g.gate_passed_V1.astype(bool)
        both = gp0 & gp1
        d = (g.loc[both, "peak_err_mm_V1"] - g.loc[both, "peak_err_mm_V0"]).dropna()
        pooled.append({"snr_db": snr, "method": m, "n": len(g), "gate_V0": gp0.mean(), "gate_V1": gp1.mean(),
                       "err_V0": g.loc[gp0, "peak_err_mm_V0"].median(), "err_V1": g.loc[gp1, "peak_err_mm_V1"].median(),
                       "acc_V0": g.loc[gp0, "region_correct_V0"].mean(), "acc_V1": g.loc[gp1, "region_correct_V1"].mean(),
                       "F1_V0": g.loc[gp0, "f1_V0"].mean(), "F1_V1": g.loc[gp1, "f1_V1"].mean(),
                       "paired_median_diff_mm": d.median() if len(d) else np.nan,
                       "wilcoxon_p": stats.wilcoxon(d).pvalue if len(d) > 5 and np.any(d != 0) else np.nan})
    show(save(pd.DataFrame(pooled), "F1_pooled"), nd=3)

    pr = P[P.bursts.astype(bool)].pivot_table(index=["snr_db"], columns="scope",
                                               values=["peak_err_mm_V1"], aggfunc="median").round(1)
    print("\nV1: median error by threshold scope (all cohorts and methods, gated or not):")
    print(pr.to_string())

    N = P[~P.bursts.astype(bool) & (P.method == F_METHODS[0])]
    nz = N.assign(dur=N.dataset.map(DUR)).groupby(["dataset", "scope"]).apply(lambda g: pd.Series({
        "gate_V0": g.gate_passed_V0.astype(bool).mean(), "gate_V1": g.gate_passed_V1.astype(bool).mean(),
        "false_bursts_per_s_V0": (g.n_detected_V0 / g.dur).mean(), "false_bursts_per_s_V1": (g.n_detected_V1 / g.dur).mean()}))
    print("\nBurst-free recordings: gate pass rate and false bursts per second, V0 vs V1:")
    show(save(nz.reset_index(), "F1_noise_only"), nd=3)

    sl = P[P.bursts.astype(bool) & (P.method == F_METHODS[0]) & (P.scope == "cohort")]
    print("\nAperiodic slope of the selected component (median) and gate failures by SNR, V0 vs V1:")
    print(sl.groupby("snr_db").agg(slope_V0=("slope_V0", "median"), slope_V1=("slope_V1", "median"),
                                   gate_fail_V0=("gate_passed_V0", lambda x: 1 - x.astype(bool).mean()),
                                   gate_fail_V1=("gate_passed_V1", lambda x: 1 - x.astype(bool).mean())).round(3).to_string())

Code check: primary fit reproduces R1, max |Δ error| = 0.00 mm (0 = identical)

  San Diego V1 pass 1: 320/320 (1067 s)
  cohort threshold 2.09
  San Diego V1 pass 2: 320/320 (1235 s)
  New Mexico V1 pass 1: 320/320 (347 s)
  cohort threshold 2.05
  New Mexico V1 pass 2: 320/320 (784 s)
  Iowa V1 pass 1: 320/320 (1011 s)
  cohort threshold 2.04
  Iowa V1 pass 2: 320/320 (1615 s)
Paired recordings × scopes × methods: 3840 (of 3840)
Primary fit (V0, 32–45 Hz flank) vs harmonic-safe fit (V1, 45–55 Hz) — cohort threshold, five motor generators, gated recordings. Paired difference V1 − V0 over recordings gated under both (negative = V1 better):
   dataset  snr_db method  n  gate_V0  gate_V1  err_V0  err_V1  acc_V0  acc_V1  F1_V0  F1_V1  paired_median_diff_mm  wilcoxon_p
      Iowa   -15.0   LCMV 50     0.74     0.74   63.47   67.52    0.14    0.08   0.17   0.16                   0.00        0.19
      Iowa   -15.0   dSPM 50     0.74     0.74   60.47   57.94    0.19    0.16   0.17   0.16    

## F2 — Real data with the harmonic-safe fit in the gate, the normalisation and the threshold

GED components are recomputed with V1 (the gate criteria are unchanged: beta peak ≥ 2 dB, slope ≤ −0.5,
sensorimotor share ≥ 0.4, no edge peak), the selected component's beta signal is divided by the V1 aperiodic amplitude,
and one common threshold per cohort is computed over the V1-included recordings. No localization.

In [4]:
F2_METRICS = ["rate_hz", "dur_ms", "time_in_burst", "env_cv", "burst_amp", "beta_peak_db", "aperiodic_slope"]


def f2_dataset(ds):
    f = REV / f"F2_real_V1_{ds}.csv"
    fi = REV / f"F2_inclusion_{ds}.csv"
    if f.exists() and fi.exists():
        return pd.read_csv(f), pd.read_csv(fi)
    t0 = time.time()
    use_fit("V1")
    try:
        D = load_real(ds)                                     # candidates evaluated with V1
        res, thr = gate_run(D, ds, GATE, {})
        chosen = {s: select(c, GATE) for s, c in D["cands"].items()}
        extra = []
        for _, r in res.iterrows():
            d = D["recs"][(r.subject, r.condition)]
            sp = spectral_v1(chosen[r.subject][0]["w"] @ d["X"], d["valid"], SFREQ)
            extra.append({"beta_peak_db": sp["beta_db"], "aperiodic_slope": sp["slope"]})
        res = pd.concat([res.reset_index(drop=True), pd.DataFrame(extra)], axis=1).assign(threshold=thr)
        inc = pd.DataFrame([{"dataset": ds, "subject": s, "group": D["recs"][(s, D["conds"][s][0])]["group"],
                             "included_V1": ok, "component_V1": c["label"]} for s, (c, ok) in chosen.items()])
    finally:
        use_fit("V0")
    res.to_csv(f, index=False)
    inc.to_csv(fi, index=False)
    print(f"  {LABEL[ds]}: {len(res)} recordings, threshold {thr:.2f} × 1/f ({time.time() - t0:.0f} s)")
    del D
    gc.collect()
    return res, inc


if RUN["F2"]:
    F2, INC = {}, []
    for ds in F_DATASETS:
        F2[ds], inc = f2_dataset(ds)
        INC.append(inc)
    INC = pd.concat(INC, ignore_index=True)

    rows = []
    for ds in F_DATASETS:
        qc = per_rec(ds, kind="detector_qc")
        i1 = INC[INC.dataset == ds]
        for grp in ("PD", "HC"):
            g1 = i1[i1.group == grp]
            row = {"dataset": LABEL[ds], "group": grp, "included_V1": f"{int(g1.included_V1.sum())}/{len(g1)}"}
            if qc is not None and "group" in qc:
                g0 = qc[(qc.group == grp) & qc.included.notna()]
                row["included_V0"] = f"{int(g0.included.astype(bool).sum())}/{len(g0)}"
            rows.append(row)
        g = i1.groupby("group").included_V1.agg(["sum", "size"])
        if {"PD", "HC"} <= set(g.index):
            tab = [[g.loc["PD", "sum"], g.loc["PD", "size"] - g.loc["PD", "sum"]],
                   [g.loc["HC", "sum"], g.loc["HC", "size"] - g.loc["HC", "sum"]]]
            orr, p = stats.fisher_exact(tab)
            rows.append({"dataset": LABEL[ds], "group": "Fisher (exclusion PD vs HC)", "included_V1": f"OR {1 / orr:.2f}, p = {p:.3f}"
                         if orr > 0 else f"p = {p:.3f}"})
    print("Participants passing the gate, primary fit (V0) vs harmonic-safe fit (V1):")
    show(save(pd.DataFrame(rows), "F2_inclusion"))

    G, Pp = [], []
    for ds in F_DATASETS:
        G.append(groups(F2[ds], F2_METRICS, ds))
        if (F2[ds].condition == "off").any():
            Pp.append(paired(F2[ds], F2_METRICS[:5], ds))
    G = save(pd.concat([g for g in G if len(g)], ignore_index=True), "F2_pd_vs_hc")
    prim = pd.read_csv(WORK / "results" / "stats_pd_hc.csv") if (WORK / "results" / "stats_pd_hc.csv").exists() else None
    if prim is not None:
        G = G.merge(prim[["dataset", "contrast", "metric", "g", "p_holm"]], on=["dataset", "contrast", "metric"],
                    how="left", suffixes=("", "_primary"))
    print("\nPD vs HC with the harmonic-safe fit (Holm over the seven metrics), next to the primary analysis:")
    show(G, ["dataset", "contrast", "metric", "n_PD", "n_HC", "PD", "HC", "g", "p", "p_holm", "g_primary", "p_holm_primary"], nd=3)
    if Pp:
        Pp = save(pd.concat([p for p in Pp if len(p)], ignore_index=True), "F2_off_on")
        print("\nOFF vs ON with the harmonic-safe fit:")
        show(Pp, ["dataset", "metric", "n_pairs", "OFF", "ON", "d_z", "p", "p_holm"], nd=3)
    chk = []
    for ds in F_DATASETS:
        r0 = per_rec(ds)
        if r0 is not None:
            m = F2[ds].merge(r0[["stem", "burst_amp", "rate_hz"]], on="stem", suffixes=("_V1", "_V0"))
            chk.append({"dataset": LABEL[ds], "recordings_both": len(m),
                        "spearman_burst_amp": stats.spearmanr(m.burst_amp_V1, m.burst_amp_V0)[0],
                        "spearman_rate": stats.spearmanr(m.rate_hz_V1, m.rate_hz_V0)[0]})
    print("\nAgreement of per-recording metrics, V1 vs primary:")
    show(pd.DataFrame(chk), nd=3)

  San Diego: 37 recordings, threshold 2.80 × 1/f (17 s)
  New Mexico: 58 recordings, threshold 3.00 × 1/f (34 s)
  Iowa: 76 recordings, threshold 2.96 × 1/f (230 s)
Participants passing the gate, primary fit (V0) vs harmonic-safe fit (V1):
   dataset                       group        included_V1 included_V0
 San Diego                          PD              14/14       14/14
 San Diego                          HC               9/11        9/11
 San Diego Fisher (exclusion PD vs HC) OR 0.00, p = 0.183         NaN
New Mexico                          PD              22/24       22/24
New Mexico                          HC              16/18       16/18
New Mexico Fisher (exclusion PD vs HC) OR 0.73, p = 1.000         NaN
      Iowa                          PD              45/87       48/87
      Iowa                          HC              31/44       32/44
      Iowa Fisher (exclusion PD vs HC) OR 2.23, p = 0.060         NaN

PD vs HC with the harmonic-safe fit (Holm over the seven me

## F3 — Random-effects inference with few cohorts: HKSJ vs modified Knapp–Hartung

With k = 2–3 cohorts and τ² = 0, the HKSJ scale factor q = Σwᵢ(yᵢ − μ̂)²/(k − 1) can be < 1 when the cohort
estimates agree more closely than their sampling variances imply; the HKSJ interval is then *narrower* than the
fixed-effect interval. The modified Knapp–Hartung method (Knapp & Hartung, 2003; Röver et al., 2015) uses
max(1, q), so the random-effects interval can never be narrower than the fixed-effect one.

In [5]:
def meta_mkh(t, effect):
    rows = []
    for m, d in t.groupby("metric", sort=False):
        d = d[np.isfinite(d[effect]) & np.isfinite(d["var"]) & (d["var"] > 0)]
        k = len(d)
        if k < 2:
            continue
        y, v = d[effect].values, d["var"].values
        w = 1 / v
        fe = np.sum(w * y) / np.sum(w)
        se_fe = 1 / np.sqrt(np.sum(w))
        q = np.sum(w * (y - fe) ** 2)
        tau2 = max(0.0, (q - (k - 1)) / (np.sum(w) - np.sum(w ** 2) / np.sum(w)))
        ws = 1 / (v + tau2)
        re = np.sum(ws * y) / np.sum(ws)
        qstar = np.sum(ws * (y - re) ** 2) / (k - 1)
        tc = stats.t.ppf(0.975, k - 1)
        se_h = np.sqrt(qstar / np.sum(ws))
        se_m = np.sqrt(max(1.0, qstar) / np.sum(ws))
        se_w = np.sqrt(1 / np.sum(ws))
        rows.append({"metric": m, "k": k, "FE": fe, "FE_low": fe - 1.96 * se_fe, "FE_high": fe + 1.96 * se_fe,
                     "FE_p": 2 * stats.norm.sf(abs(fe / se_fe)), "tau2": tau2, "q_scale": qstar, "RE": re,
                     "RE_wald_p": 2 * stats.norm.sf(abs(re / se_w)),
                     "HKSJ_low": re - tc * se_h, "HKSJ_high": re + tc * se_h,
                     "HKSJ_p": 2 * stats.t.sf(abs(re / se_h), k - 1) if se_h > 0 else np.nan,
                     "mKH_low": re - tc * se_m, "mKH_high": re + tc * se_m, "mKH_p": 2 * stats.t.sf(abs(re / se_m), k - 1)})
    return pd.DataFrame(rows)


if RUN["F3"]:
    rs = lambda n: pd.read_csv(WORK / "results" / f"stats_{n}.csv") if (WORK / "results" / f"stats_{n}.csv").exists() else pd.DataFrame()
    ph, ob, om = rs("pd_hc"), rs("offon_bursts"), rs("offon_maps")
    sets = []
    if len(ph):
        sets.append(meta_mkh(ph[ph.contrast == "PD-ON vs HC"], "g").assign(analysis="PD-ON vs HC (primary; Table 12)"))
        if (ph.contrast == "PD-OFF vs HC").any():
            sets.append(meta_mkh(ph[ph.contrast == "PD-OFF vs HC"], "g").assign(analysis="PD-OFF vs HC (primary)"))
    if len(ob):
        sets.append(meta_mkh(ob, "d_z").assign(analysis="OFF vs ON bursts (primary)"))
    if len(om):
        sets.append(meta_mkh(om, "d_z").assign(analysis="OFF vs ON maps (primary)"))
    if RUN["F2"] and len(G):
        sets.append(meta_mkh(G[G.contrast == "PD-ON vs HC"], "g").assign(analysis="PD-ON vs HC (V1 fit)"))
        if len(Pp):
            sets.append(meta_mkh(Pp, "d_z").assign(analysis="OFF vs ON bursts (V1 fit)"))
    g4 = REV / "R4_gate_sensitivity.csv"
    if g4.exists():
        G4 = pd.read_csv(g4)
        x = G4[(G4.analysis == "PD vs HC") & (G4.contrast == "PD-ON vs HC") & (G4.metric == "burst_amp")]
        gm = [meta_mkh(d.assign(metric=gate), "g") for gate, d in x.groupby("gate", sort=False)]
        if gm:
            sets.append(pd.concat(gm).assign(analysis="PD-ON vs HC burst amplitude, per gate (Table S2)"))
    w1 = REV / "W1_pd_vs_hc.csv"
    if w1.exists():
        W1 = pd.read_csv(w1)
        sets.append(meta_mkh(W1[(W1.contrast == "PD-ON vs HC") & W1.metric.str.match(r"(burst_amp|slope)_V\d")], "g")
                    .assign(analysis="PD-ON vs HC, aperiodic models (Table 14 / text)"))
    M3 = save(pd.concat(sets, ignore_index=True), "F3_meta_mkh")
    print("Random effects (DerSimonian–Laird τ²) with HKSJ and modified Knapp–Hartung (mKH) 95% CIs; "
          "q_scale < 1 is where HKSJ is narrower than the fixed effect:")
    for an, d in M3.groupby("analysis", sort=False):
        print(f"\n{an}")
        show(d, ["metric", "k", "FE", "FE_low", "FE_high", "FE_p", "tau2", "q_scale", "RE", "HKSJ_low", "HKSJ_high",
                 "HKSJ_p", "mKH_low", "mKH_high", "mKH_p"], nd=3)

Random effects (DerSimonian–Laird τ²) with HKSJ and modified Knapp–Hartung (mKH) 95% CIs; q_scale < 1 is where HKSJ is narrower than the fixed effect:

PD-ON vs HC (primary; Table 12)
         metric  k     FE  FE_low  FE_high  FE_p  tau2  q_scale     RE  HKSJ_low  HKSJ_high  HKSJ_p  mKH_low  mKH_high  mKH_p
        rate_hz  3 -0.334  -0.673    0.006 0.054 0.000    0.525 -0.334    -0.874      0.206   0.117   -1.079     0.411  0.194
         dur_ms  3 -0.263  -0.602    0.077 0.129 0.045    0.918 -0.217    -1.116      0.682   0.408   -1.156     0.721  0.424
  time_in_burst  3 -0.345  -0.685   -0.006 0.046 0.000    0.557 -0.345    -0.902      0.211   0.116   -1.090     0.400  0.184
         env_cv  3 -0.301  -0.643    0.042 0.085 0.266    0.826 -0.183    -1.566      1.199   0.626   -1.704     1.338  0.656
      burst_amp  3 -0.483  -0.827   -0.139 0.006 0.126    0.852 -0.399    -1.501      0.703   0.260   -1.593     0.795  0.287
     snr_est_db  3 -0.431  -0.776   -0.087 0.014 0.275    0.

## F4 — Two simultaneous generators: does "both imaged" depend on the criterion?

R3's two-generator cells (left + right M1, independent or synchronous bursts, San Diego montage, −5 and 0 dB,
the same seeds) are rerun and the second source is counted as imaged for every combination of level
(local maximum within the radius ≥ level × global maximum, above the LCMV baseline of 1) and radius.
A map with two separate peaks is also counted directly: the second-highest local maximum (≥ 30 mm from the first)
lies within 20 mm of the other source and reaches ≥ 50 % of the global maximum.

In [6]:
def second_peak(mp, a, base, min_sep=30.0):
    j = int(np.argmax(mp))
    far = np.linalg.norm(a["pos_inv"] - a["pos_inv"][j], axis=1) >= min_sep
    if not far.any():
        return j, None, 0.0
    j2 = int(np.flatnonzero(far)[np.argmax(mp[far])])
    return j, j2, (mp[j2] - base) / max(mp[j] - base, 1e-12)


if RUN["F4"] and R3_DATASET in ANAT:
    f4 = REV / "F4_two_generators.csv"
    if f4.exists():
        T4 = pd.read_csv(f4)
    else:
        a0, cfg0 = ANAT[R3_DATASET], {**CFG, "duration": DUR[R3_DATASET]}
        gL, gR = a0["regions"]["L_M1_hand"]["gain"], a0["regions"]["R_M1_hand"]["gain"]
        cL, cR = a0["regions"]["L_M1_hand"]["centroid"], a0["regions"]["R_M1_hand"]["centroid"]
        seeds = lambda k: [810_000 + 100 * k + s for s in range(R3_SEEDS)]
        rows, t0 = [], time.time()
        for snr in R3_SNRS:
            def second(X, rng, snr=snr):
                g = gR - gR.mean()
                s2, _ = sim.make_source(rng, X.shape[1], SFREQ, CFG["conditions"]["OFF"], cfg0)
                best = int(np.argmax(np.abs(g)))
                bg = np.std(sim.bandpass(X[best], CFG["beta_band"], SFREQ))
                inb = s2 != 0
                Y = X + 10 ** (snr / 20) * bg / np.sqrt(np.mean((g[best] * s2[inb]) ** 2)) * np.outer(g, s2)
                return Y - Y.mean(axis=0, keepdims=True)
            for kind, k, kw in (("synchronous", 20, {"gain_override": gL + gR}), ("independent", 21, {"extra": second})):
                for seed in seeds(k):
                    X, ev, v = sim_record(a0, "L_M1_hand", snr, seed, cfg0, **kw)
                    det = choose_detector([X], [v], SFREQ, a0["ch_names"], gate=True)
                    sb, _ = normalised_beta(X, v, det["w"], SFREQ)
                    thr = np.percentile(np.abs(signal.hilbert(sb))[v], THRESH_PERCENTILE)
                    evd, _ = detect_real(sb, v, SFREQ, thr)
                    maps, _ = localize_real(prepare(X, v, SFREQ), evd, a0["G_inv"], F_METHODS)
                    for m, mp in maps.items():
                        if not np.all(np.isfinite(mp)):
                            continue
                        base = 1.0 if m == "LCMV" else 0.0
                        j, j2, rel2 = second_peak(mp, a0, base)
                        dL, dR = (np.linalg.norm(a0["pos_inv"][j] - c) for c in (cL, cR))
                        other = cR if dL <= dR else cL
                        row = {"snr_db": snr, "kind": kind, "seed": seed, "method": m, "gate_passed": det["passed"],
                               "err_nearer_mm": min(dL, dR), "peak_hemi": "L" if dL <= dR else "R",
                               "second_peak_near_other": (j2 is not None and
                                                          np.linalg.norm(a0["pos_inv"][j2] - other) < 20 and rel2 >= 0.5)}
                        for rad in F4_RADII:
                            near = np.linalg.norm(a0["pos_inv"] - other, axis=1) < rad
                            rel = (mp[near].max() - base) / max(mp[j] - base, 1e-12)
                            for lev in F4_LEVELS:
                                row[f"both_{int(lev * 100)}pct_{int(rad)}mm"] = rel >= lev
                        rows.append(row)
                    print(f"\r  {len(rows)} maps ({time.time() - t0:.0f} s)", end="")
        print()
        T4 = pd.DataFrame(rows)
        T4.to_csv(f4, index=False)
    bc = [c for c in T4.columns if c.startswith("both_")] + ["second_peak_near_other"]
    S4 = T4.groupby(["kind", "snr_db", "method"])[bc + ["err_nearer_mm"]].agg(
        {**{c: "mean" for c in bc}, "err_nearer_mm": "median"}).reset_index()
    print("Fraction of recordings in which the second generator is imaged, by criterion (level of the global maximum, radius):")
    show(save(S4, "F4_summary"), nd=2)
    print("\nPeak hemisphere (L = the generator in the main region):")
    print(T4.groupby(["kind", "method"]).peak_hemi.value_counts(normalize=True).unstack(fill_value=0).round(2).to_string())

  80 maps (147 s)
Fraction of recordings in which the second generator is imaged, by criterion (level of the global maximum, radius):
       kind  snr_db method  both_30pct_15mm  both_50pct_15mm  both_70pct_15mm  both_30pct_20mm  both_50pct_20mm  both_70pct_20mm  both_30pct_25mm  both_50pct_25mm  both_70pct_25mm  second_peak_near_other  err_nearer_mm
independent    -5.0   LCMV              0.2              0.2              0.0              0.2              0.2              0.1              0.2              0.2              0.1                     0.1          14.80
independent    -5.0   dSPM              0.7              0.4              0.1              0.7              0.4              0.1              0.8              0.4              0.1                     0.0          24.21
independent     0.0   LCMV              0.1              0.0              0.0              0.1              0.0              0.0              0.1              0.0              0.0                     0.0      